In [ ]:
import os
import numpy as np
import keras
from keras import layers, regularizers, callbacks
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from collections import defaultdict
import pandas as pd
from sklearn.model_selection import train_test_split
import pandas as pd
from joblib import Parallel, delayed
from tensorflow.keras import layers, regularizers, callbacks

In [ ]:
# Load dataset
df = pd.read_csv("data.csv")

X = df[['LA', 'SAl', 'SAz', 'DNI', 'DHI', 'GHI']].values
y = df[['Ev', 'rUDI']].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Normalize features and targets
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

In [ ]:
# Ev normalization options
def normalize_ev(y_train_ev, y_test_ev, method):
    if method == 'standard':
        scaler = StandardScaler()
        return scaler.fit_transform(y_train_ev), scaler.transform(y_test_ev), scaler.inverse_transform
    elif method == 'yeo-johnson':
        pt = PowerTransformer(method='yeo-johnson')
        y_train_pt = pt.fit_transform(y_train_ev)
        y_test_pt = pt.transform(y_test_ev)
        scaler = StandardScaler()
        return scaler.fit_transform(y_train_pt), scaler.transform(y_test_pt), lambda x: pt.inverse_transform(scaler.inverse_transform(x))
    elif method == 'log':
        y_train_log = np.log1p(y_train_ev)
        y_test_log = np.log1p(y_test_ev)
        scaler = StandardScaler()
        return scaler.fit_transform(y_train_log), scaler.transform(y_test_log), lambda x: np.expm1(scaler.inverse_transform(x))
    else:
        raise ValueError("Unknown normalization method")

# Hyperparameters
hidden_layers = [
    [512, 256, 128], [256, 128], 
    [512, 256, 64], [256, 128, 64],
    [1024, 512, 256]
]
dropouts = [(0.05, 0.2), (0.1, 0.3)]
batch_sizes = [64, 128]
lr_range = [0.001, 0.005]
l2_range = [0.001, 0.01]
ev_normalizers = ['standard', 'yeo-johnson', 'log']

# Results storage
results = defaultdict(list)
best_r2_ev = -np.inf
best_model = None
best_config = None
model_count = 0

# Train model and evaluate its performance
def train_and_evaluate_model(ev_norm, hidden, dropout1, dropout2, batch_size, lr, l2_reg, model_count):
    global best_r2_ev, best_model, best_config
    config_str = f"_EV{ev_norm}_HL{hidden}_dr1{dropout1}_dr2{dropout2}_bs{batch_size}_lr{lr}_l2{l2_reg}"
    print(f"\nTraining Model {model_count}/240+: {config_str}")

    # Normalize targets
    y_train_ev_scaled, y_test_ev_scaled, inverse_ev = normalize_ev(
        y_train[:, 0].reshape(-1, 1), y_test[:, 0].reshape(-1, 1), ev_norm
    )
    scaler_rudi = StandardScaler()
    y_train_rudi_scaled = scaler_rudi.fit_transform(y_train[:, 1].reshape(-1, 1))
    y_test_rudi_scaled = scaler_rudi.transform(y_test[:, 1].reshape(-1, 1))

    y_train_scaled = np.column_stack([y_train_ev_scaled, y_train_rudi_scaled])
    y_test_scaled = np.column_stack([y_test_ev_scaled, y_test_rudi_scaled])

    # Model
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_train_scaled.shape[1],)))
    for i, units in enumerate(hidden):
        model.add(layers.Dense(units, activation='relu', kernel_regularizer=regularizers.l2(l2_reg)))
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(dropout1 if i == 0 else dropout2))
    model.add(layers.Dense(2)) 

    
    # Compile
    optimizer = keras.optimizers.Adam(
        learning_rate=keras.optimizers.schedules.ExponentialDecay(
            initial_learning_rate=lr,
            decay_steps=100,
            decay_rate=0.9
        )
    )
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

    # Train
    early_stop = callbacks.EarlyStopping(patience=25, restore_best_weights=True, monitor='val_loss')
    history = model.fit(
        X_train_scaled, y_train_scaled,
        epochs=300,
        batch_size=batch_size,
        validation_data=(X_test_scaled, y_test_scaled),
        callbacks=[early_stop],
        verbose=0
    )

    # Predict
    y_pred_scaled = model.predict(X_test_scaled)
    y_pred_ev = inverse_ev(y_pred_scaled[:, 0].reshape(-1, 1)).flatten()
    y_pred_rudi = scaler_rudi.inverse_transform(y_pred_scaled[:, 1].reshape(-1, 1)).flatten()
    y_pred = np.column_stack([y_pred_ev, y_pred_rudi])

    # Evaluate
    r2_ev = r2_score(y_test[:, 0], y_pred[:, 0])
    r2_rudi = r2_score(y_test[:, 1], y_pred[:, 1])
    r2_model = r2_score(y_test, y_pred)
    mae_ev = mean_absolute_error(y_test[:, 0], y_pred[:, 0])
    mae_rudi = mean_absolute_error(y_test[:, 1], y_pred[:, 1])
    mae_model = mean_absolute_error(y_test, y_pred)
    mse_ev = mean_squared_error(y_test[:, 0], y_pred[:, 0])
    mse_rudi = mean_squared_error(y_test[:, 1], y_pred[:, 1])
    mse_model = mean_squared_error(y_test, y_pred)

    # Save model
    model_filename = f"saved_models/{config_str}.h5"
    model.save(model_filename)
    
    return {
    'config': config_str,
    'r2_ev': r2_ev,
    'r2_rudi': r2_rudi,
    'r2_model': r2_model,
    'mae_ev': mae_ev,
    'mae_rudi': mae_rudi,
    'mae_model': mae_model,
    'mse_ev': mse_ev,
    'mse_rudi': mse_rudi,
    'mse_model': mse_model
    }



tasks = []
for ev_norm in ev_normalizers:
    for hidden in hidden_layers:
        for dropout1, dropout2 in dropouts:
            for batch_size in batch_sizes:
                for lr in lr_range:
                    for l2_reg in l2_range:
                        model_count += 1
                        tasks.append(delayed(train_and_evaluate_model)(
                            ev_norm, hidden, dropout1, dropout2,
                            batch_size, lr, l2_reg, model_count
                        ))

# Run the tasks and collect results
results_list = Parallel(n_jobs=-1, verbose=10)(tasks)

results = defaultdict(list)
for res in results_list:
    for key, value in res.items():
        results[key].append(value)

# Save results
print(f"Results saved to architecture_comparison.xlsx")
print(f"All models saved in /saved_models/")
print(f"Best model (R² for Ev = {best_r2_ev:.4f}): {best_config}")

results_df = pd.DataFrame(results)
results_df.to_excel("architecture_comparison.xlsx", index=False)
results_df = pd.DataFrame(results)
results_df.to_csv("architecture_comparison.csv", index=False)
